# 01 · Problem Understanding

## Objective
Understand the business problem, define the prediction target, and get a first look at the
dataset's structure before any analysis or modelling.

## 1. Business problem

A telecom company earns recurring monthly revenue from subscribers. When a customer **churns**
(cancels their service), that revenue stream stops. Retaining an existing customer is generally
considered cheaper than acquiring a new one, so the company wants to **identify customers who are
likely to churn early enough to intervene** (e.g. offer a discount, a contract upgrade or better support).

**Data Science framing**

| Item | Definition |
|---|---|
| Problem type | Supervised **binary classification** |
| Target | `Churn` — `Yes` if the customer left within the last month, else `No` |
| Unit of analysis | One row = one customer |
| Output we want | A churn **probability** per customer, so the business can rank and prioritise |
| Main users | Retention / marketing team |

**Why the output should be a probability, not just Yes/No:** a retention budget is limited.
Ranking customers by risk lets the team target the top-N most at-risk customers, and the decision
threshold can be adjusted to match the budget.

## 2. Setup

In [1]:
import sys
from pathlib import Path

# Make the project root importable so we can use src/churn_utils.py
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from src.churn_utils import RAW_DATA_PATH

pd.set_option("display.max_columns", None)

## 3. Load the dataset

The raw file is `data/data.csv`. This is the **IBM Telco Customer Churn** sample dataset
(widely distributed on Kaggle as *"Telco Customer Churn"*, `WA_Fn-UseC_-Telco-Customer-Churn.csv`).

In [2]:
df = pd.read_csv(RAW_DATA_PATH)
print("Shape:", df.shape)
df.head()

Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## 4. Understanding the features

Column meanings below follow the dataset's public documentation; they were cross-checked against the
actual values in the file (next cell).

| Group | Column | Meaning |
|---|---|---|
| ID | `customerID` | Unique customer identifier |
| Demographics | `gender` | Male / Female |
| | `SeniorCitizen` | 1 if the customer is 65+, else 0 |
| | `Partner` | Has a partner (Yes/No) |
| | `Dependents` | Has dependents (Yes/No) |
| Account | `tenure` | Number of **months** the customer has stayed with the company |
| | `Contract` | Month-to-month / One year / Two year |
| | `PaperlessBilling` | Uses paperless billing (Yes/No) |
| | `PaymentMethod` | Electronic check / Mailed check / Bank transfer (automatic) / Credit card (automatic) |
| | `MonthlyCharges` | Current amount charged per month |
| | `TotalCharges` | Total amount charged over the customer's lifetime |
| Phone services | `PhoneService` | Has phone service |
| | `MultipleLines` | Has multiple lines (or "No phone service") |
| Internet services | `InternetService` | DSL / Fiber optic / No |
| | `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` | Add-on services (Yes / No / "No internet service") |
| **Target** | `Churn` | Left within the last month (Yes/No) |

In [3]:
# Quick check: do the actual values match the documented meanings?
for col in df.columns.drop(["customerID", "tenure", "MonthlyCharges", "TotalCharges"]):
    print(f"{col:<17} {sorted(df[col].astype(str).unique())}")

gender            ['Female', 'Male']
SeniorCitizen     ['0', '1']
Partner           ['No', 'Yes']
Dependents        ['No', 'Yes']
PhoneService      ['No', 'Yes']
MultipleLines     ['No', 'No phone service', 'Yes']
InternetService   ['DSL', 'Fiber optic', 'No']
OnlineSecurity    ['No', 'No internet service', 'Yes']
OnlineBackup      ['No', 'No internet service', 'Yes']
DeviceProtection  ['No', 'No internet service', 'Yes']
TechSupport       ['No', 'No internet service', 'Yes']
StreamingTV       ['No', 'No internet service', 'Yes']
StreamingMovies   ['No', 'No internet service', 'Yes']
Contract          ['Month-to-month', 'One year', 'Two year']
PaperlessBilling  ['No', 'Yes']
PaymentMethod     ['Bank transfer (automatic)', 'Credit card (automatic)', 'Electronic check', 'Mailed check']
Churn             ['No', 'Yes']


In [4]:
df[["tenure", "MonthlyCharges"]].agg(["min", "max"])

,tenure,MonthlyCharges
min,0,18.25
max,72,118.75


**Observations**
- Values match the documented meanings: binary columns are Yes/No, `SeniorCitizen` is 0/1,
  service columns use the extra levels "No internet service" / "No phone service".
- `tenure` ranges from 0 to 72 months; `MonthlyCharges` from 18.25 to 118.75.
- `TotalCharges` is stored as **text** although it should be numeric — flagged for notebook 02/03.

## 5. Target variable and business significance

In [5]:
churn_counts = df["Churn"].value_counts()
churn_rate = df["Churn"].eq("Yes").mean()
print(churn_counts.to_string())
print(f"\nChurn rate: {churn_rate:.1%}")

# How much monthly revenue is attached to customers who churned?
monthly_revenue = df.groupby("Churn")["MonthlyCharges"].sum()
print(f"\nMonthly revenue from churned customers: {monthly_revenue['Yes']:,.0f} "
      f"({monthly_revenue['Yes'] / monthly_revenue.sum():.1%} of total monthly charges)")

Churn
No     5174
Yes    1869

Churn rate: 26.5%

Monthly revenue from churned customers: 139,131 (30.5% of total monthly charges)


**Observations**
- 1,869 of 7,043 customers churned (**26.5%**): the classes are imbalanced (~1 churner per 3 retained).
- Churners account for **30.5% of total monthly charges** — a larger share than their head-count share,
  so churners tend to be on more expensive plans. This makes the problem financially important.

## 6. Important considerations before starting

- **Snapshot data:** there is no date column, so we cannot do a time-based split. A random,
  **stratified** train/test split is the reasonable choice.
- **Leakage check:** none of the columns is recorded *after* the churn event (e.g. no "cancellation
  reason" column), so all features are legitimately available at prediction time.
  `TotalCharges` grows with tenure, which is fine — it is known before the customer churns.
- **`customerID`** is an identifier, not a behaviour; it will be removed before modelling.
- **Metric choice:** because churners are the minority class, accuracy alone will be misleading.
  We will focus on **recall / precision / F1 for churners** and **ROC-AUC** (ranking quality).

## Conclusion
- The task is binary classification of `Churn` with 7,043 customers and 19 usable features
  (plus an ID and the target).
- Features cover demographics, account/billing information and subscribed services.
- The next notebook performs a basic inspection of data types, missing values and data quality.